# 4.0 Churn Prediction: Evaluation and Interpretation

**Goal:** predict which customers will not buy again in the 90 days after the cutoff date (10 Sep 2011), and judge how good and how useful the model is.

**Data:** `data/processed/customer_churn_features.csv`. Features use only purchases *before* the cutoff and the label uses purchases *after* it, so there is no leakage.

**Contents**
1. Load data and split (same split as `src/modeling/train.py`)
2. Baseline: how well does one feature (recency) do alone?
3. Model comparison, ROC curves, confusion matrix
4. Precision vs recall and choosing a threshold
5. Which features matter
6. Hyperparameter tuning (GridSearchCV)
7. Out-of-fold predictions and risk tiers for the dashboard

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split

warnings.filterwarnings("ignore")

# Work from the repo root so the project's relative paths resolve.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.modeling.predict import assign_risk_tier
from src.modeling.train import RANDOM_STATE, build_models, load_data

FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## 1. Load data and split

In [2]:
X, y = load_data()
print(f"Customers: {len(X):,} | Features: {X.shape[1]} | Churn rate: {y.mean():.1%}")
print("Features:", ", ".join(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Churn rate train/test: {y_train.mean():.1%} / {y_test.mean():.1%}")

Customers: 5,256 | Features: 19 | Churn rate: 56.4%
Features: recency_days, frequency, monetary_total, monetary_avg, quantity_total, quantity_avg, unique_products, tenure_days, purchase_span_days, cancellation_count, cancellation_rate, refund_amount, orders_30d, spend_30d, orders_60d, spend_60d, is_uk, order_velocity_days, momentum_spend_ratio
Train: 4,204 | Test: 1,052 | Churn rate train/test: 56.4% / 56.5%


The split is stratified, so train and test keep the same churn rate. It is a random split of customers; time leakage is already prevented earlier, because features come from before the cutoff and labels from after it.

## 2. Baseline: recency alone

In [3]:
recency_auc = roc_auc_score(y_test, X_test["recency_days"])
print(f"ROC-AUC using only recency_days: {recency_auc:.3f}")

ROC-AUC using only recency_days: 0.762


**What this shows:** one feature, days since last purchase, already reaches ROC-AUC 0.762, which on its own meets the 0.75 target. So the model has to be judged against 0.762, not against 0.5 (random guessing). Recent buyers tend to keep buying, which is the intuition behind churn.

## 3. Model comparison

In [4]:
models = build_models()
for model in models.values():
    model.fit(X_train, y_train)

proba = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}

rows = []
for name, p in proba.items():
    pred = (p >= 0.5).astype(int)
    rows.append({
        "Model": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, p),
    })
rows.append({"Model": "Baseline: recency only", "Precision": np.nan, "Recall": np.nan,
             "F1": np.nan, "ROC-AUC": recency_auc})
results = pd.DataFrame(rows).set_index("Model").round(3)
results

,Precision,Recall,F1,ROC-AUC
Model,,,,
Logistic Regression,0.761,0.726,0.743,0.791
Random Forest,0.756,0.731,0.743,0.794
Baseline: recency only,NaN,NaN,NaN,0.762


In [5]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, p in proba.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
fpr, tpr, _ = roc_curve(y_test, X_test["recency_days"])
ax.plot(fpr, tpr, linestyle="--", label=f"Recency only (AUC {recency_auc:.3f})")
ax.plot([0, 1], [0, 1], color="grey", linestyle=":", label="Random guess")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate (recall)")
ax.set_title("ROC curves on the test set")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "churn_roc_curves.png", dpi=150)
plt.show()

**What this shows:** both models beat the recency-only line, but only by about 0.03 AUC (0.791 and 0.794 against 0.762). The extra 18 features help a little; they do not change the picture. The two models are almost identical, so the simpler Logistic Regression is a fair alternative if explainability matters most.

In [6]:
best_name = max(proba, key=lambda n: roc_auc_score(y_test, proba[n]))
pred_best = (proba[best_name] >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_best)
tn, fp, fn, tp = cm.ravel()
print(f"Best model: {best_name}")
print(f"True negatives (stayed, predicted stayed): {tn}")
print(f"False positives (stayed, predicted churn): {fp}")
print(f"False negatives (churned, predicted stayed): {fn}")
print(f"True positives (churned, predicted churn): {tp}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["Stayed", "Churned"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion matrix: {best_name} (threshold 0.5)")
fig.tight_layout()
fig.savefig(FIG_DIR / "churn_confusion_matrix.png", dpi=150)
plt.show()

Best model: Random Forest
True negatives (stayed, predicted stayed): 318
False positives (stayed, predicted churn): 140
False negatives (churned, predicted stayed): 160
True positives (churned, predicted churn): 434


**What this shows (Random Forest, threshold 0.5, 1,052 test customers):** of the 594 customers who really churned, 434 were caught (recall 73.1%) and 160 were missed. Of the 574 customers flagged, 434 really churned (precision 75.6%) and 140 would have stayed.

## 4. Precision vs recall and the threshold

A model outputs a churn *probability*. The 0.5 cut-off is only a default. Moving it trades two errors against each other:

- **Recall** = of customers who really churn, how many do we catch? Missing one means losing a customer we could have tried to keep.
- **Precision** = of customers we flag, how many really churn? A low value means retention offers go to people who would have stayed anyway.

For a retention campaign where an offer is cheap and a lost customer is expensive, **recall usually matters more**, so a lower threshold can make sense.

In [7]:
p = proba[best_name]
thresholds = np.arange(0.30, 0.81, 0.05)
rows = []
for t in thresholds:
    pred = (p >= t).astype(int)
    rows.append({
        "Threshold": round(t, 2),
        "Flagged customers": int(pred.sum()),
        "Flagged %": round(pred.mean() * 100, 1),
        "Precision": round(precision_score(y_test, pred), 3),
        "Recall": round(recall_score(y_test, pred), 3),
        "F1": round(f1_score(y_test, pred), 3),
    })
threshold_table = pd.DataFrame(rows).set_index("Threshold")
threshold_table

,Flagged customers,Flagged %,Precision,Recall,F1
Threshold,,,,,
0.30,798,75.9,0.685,0.921,0.786
0.35,731,69.5,0.709,0.872,0.782
0.40,685,65.1,0.723,0.833,0.774
0.45,637,60.6,0.739,0.793,0.765
0.50,574,54.6,0.756,0.731,0.743
0.55,511,48.6,0.779,0.670,0.720
0.60,435,41.3,0.800,0.586,0.676
0.65,367,34.9,0.828,0.512,0.633
0.70,292,27.8,0.832,0.409,0.549


In [8]:
prec, rec, thr = precision_recall_curve(y_test, p)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(thr, prec[:-1], label="Precision")
ax.plot(thr, rec[:-1], label="Recall")
ax.axvline(0.5, color="grey", linestyle=":", label="Default 0.5")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title(f"Precision and recall vs threshold: {best_name}")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "churn_threshold_analysis.png", dpi=150)
plt.show()

**What this shows:** lowering the threshold from 0.50 to 0.40 raises recall from 0.731 to 0.833, so 100 more churners are caught per 594, but precision falls from 0.756 to 0.723 and the share of customers flagged rises from 54.6% to 65.1%. Whether that is worth it depends on the cost of a retention offer against the value of a customer, which is a business decision, not a statistical one. No cost figures exist in this dataset, so 0.50 is kept as the reported default and the table above gives the alternatives.

## 5. Which features matter

In [9]:
rf = models["Random Forest"]
perm = permutation_importance(
    rf, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
importance = (
    pd.DataFrame({"Permutation importance (AUC drop)": perm.importances_mean,
                  "Std": perm.importances_std}, index=X.columns)
    .sort_values("Permutation importance (AUC drop)", ascending=False)
)
top = importance.head(10)
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(top.index[::-1], top["Permutation importance (AUC drop)"][::-1],
        xerr=top["Std"][::-1], color="#3b6ea5")
ax.set_xlabel("Drop in test ROC-AUC when the feature is shuffled")
ax.set_title("Top 10 features (Random Forest, permutation importance)")
fig.tight_layout()
fig.savefig(FIG_DIR / "churn_feature_importance.png", dpi=150)
plt.show()
importance.head(10).round(4)

,Permutation importance (AUC drop),Std
recency_days,0.0361,0.0065
unique_products,0.0084,0.0019
order_velocity_days,0.0066,0.0027
spend_60d,0.0050,0.0020
monetary_total,0.0031,0.0037
frequency,0.0015,0.0014
tenure_days,0.0013,0.0021
momentum_spend_ratio,0.0011,0.0010
orders_60d,0.0008,0.0008
purchase_span_days,0.0006,0.0029


**What this shows:** `recency_days` dominates. Shuffling it drops test AUC by 0.036, more than four times the next feature (`unique_products`, 0.008). Most other features contribute almost nothing, and several have a spread (Std) as large as their value, so their ranking is not reliable. This agrees with the baseline result: the model is mostly a recency model.

## 6. Hyperparameter tuning

In [10]:
grid = GridSearchCV(
    estimator=build_models()["Random Forest"],
    param_grid={"max_depth": [6, 10, None], "min_samples_leaf": [3, 5, 10]},
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)
grid.fit(X_train, y_train)
tuned_auc = roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1])
default_auc = roc_auc_score(y_test, proba["Random Forest"])
print("Best parameters:", grid.best_params_)
print(f"Best cross-validated ROC-AUC (train only): {grid.best_score_:.3f}")
print(f"Test ROC-AUC, default Random Forest: {default_auc:.3f}")
print(f"Test ROC-AUC, tuned Random Forest:   {tuned_auc:.3f}")

Best parameters: {'max_depth': 6, 'min_samples_leaf': 10}
Best cross-validated ROC-AUC (train only): 0.805
Test ROC-AUC, default Random Forest: 0.794
Test ROC-AUC, tuned Random Forest:   0.801


**What this shows:** the best settings were `max_depth=6` and `min_samples_leaf=10`, which is a more restricted forest than the default. Cross-validated AUC is 0.805 and test AUC rises from 0.794 to 0.801. That is a gain of 0.007 on 1,052 test customers, small enough to be partly noise, so treat tuning as a mild improvement rather than a breakthrough. This notebook does not overwrite `models/churn_model.pkl`; the saved model is still the default Random Forest.

## 7. Out-of-fold predictions and risk tiers

For the dashboard, every customer needs a churn probability. If the final model scored the customers it was trained on, the numbers would look better than they are. Instead each customer is scored by a model that **never saw that customer** (5-fold out-of-fold predictions).

Risk tiers come from `assign_risk_tier` in `src/modeling/predict.py`, so the notebook, the scoring script and the dashboard all use one definition: **High Risk** at 0.70 or above, **Medium Risk** at 0.40 or above, otherwise **Low Risk**.

In [11]:
ids = pd.read_csv("data/processed/customer_churn_features.csv")["customer_id"]
oof = cross_val_predict(
    build_models()["Random Forest"], X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    method="predict_proba", n_jobs=-1,
)[:, 1]

predictions = pd.DataFrame({"customer_id": ids, "churn_probability": oof.round(4), "actual_churn": y.values})
predictions["risk_tier"] = [assign_risk_tier(p) for p in oof]
print(f"Out-of-fold ROC-AUC across all customers: {roc_auc_score(y, oof):.3f}")
tier_summary = predictions.groupby("risk_tier").agg(
    customers=("customer_id", "count"), actual_churn_rate=("actual_churn", "mean")
).reindex(["Low Risk", "Medium Risk", "High Risk"])
tier_summary["actual_churn_rate"] = (tier_summary["actual_churn_rate"] * 100).round(1)
tier_summary

Out-of-fold ROC-AUC across all customers: 0.797


,customers,actual_churn_rate
risk_tier,,
Low Risk,1702,24.6
Medium Risk,1910,61.5
High Risk,1644,83.6


In [12]:
top_decile = predictions.nlargest(int(len(predictions) * 0.10), "churn_probability")
print(f"Overall churn rate: {y.mean():.1%}")
print(f"Churn rate in the top 10% highest-risk customers: {top_decile['actual_churn'].mean():.1%}")
print(f"Lift: {top_decile['actual_churn'].mean() / y.mean():.2f}x")

predictions.sort_values("churn_probability", ascending=False).to_csv(
    "data/processed/churn_predictions.csv", index=False
)
print(f"Saved {len(predictions):,} rows to data/processed/churn_predictions.csv")

Overall churn rate: 56.4%
Churn rate in the top 10% highest-risk customers: 88.8%
Lift: 1.57x
Saved 5,256 rows to data/processed/churn_predictions.csv


**What this shows:** the tiers separate customers well. Actual churn is 24.6% in Low, 61.5% in Medium and 83.6% in High, so higher predicted risk really means higher churn. The 10% of customers the model ranks highest churn at 88.8%, compared with 56.4% overall. That is a lift of 1.57x; the best possible at this churn rate would be 1.77x. These predictions are saved in `data/processed/churn_predictions.csv` for the dashboard's high-risk list.

## Conclusion

- **Target met:** test ROC-AUC is 0.794 (default Random Forest) and 0.801 (tuned), above the 0.75 requirement. Out-of-fold AUC over all customers is 0.797.
- **Honest reading:** recency alone scores 0.762, so most of the predictive power comes from one feature. The full model adds about 0.03 to 0.04 AUC.
- **Useful for ranking:** risk tiers match real churn rates (24.6%, 61.5%, 83.6%), so the probabilities can drive a prioritised retention list.
- **Limits to state in the report:**
  - The churn window is 90 days and covers the Sep to Dec holiday peak, so churn (56.4%) is high. A 30 or 60 day window is untested.
  - There is one cutoff date. Results at other cutoffs are unknown.
  - The threshold is not tuned to costs, because the dataset has none.
  - The tuned model (AUC 0.801) is not yet the saved model.
- **Next:** decide whether to save the tuned model, then feed `churn_predictions.csv` and the segments into the dashboard.